# Model Training

## Import necessary libraries

In [1]:
%pip install -qq -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Add current directory to Python path for imports
import os
import sys

# Add the parent directory (project root) to Python path so we can import from src
project_root = os.path.dirname(os.getcwd())
if project_root not in sys.path:
    sys.path.append(project_root)

In [3]:
# Utility Functions
from src.utils import create_spark_session

# Create Spark session
spark, sedona = create_spark_session(app_name="ModelTrainingSpark")

## Loading Datasets

In [4]:
from src.utils import read_config_path

# Load data using configuration file
filepath = read_config_path(key="raw_data_path")

df = spark.read.csv(
    filepath,
    header=True,
    inferSchema=True,
    multiLine=True,
    escape='"',
    quote='"',
)

df.show(10)

+-----------+-------------------+--------------------+--------------------+--------------------+--------------------+------------------+--------------------+-----------+--------+-------------+--------------------+---------+----+------------+--------------------+
|  ticket_id|               type|        organization|             comment|               photo|         photo_after|            coords|             address|subdistrict|district|     province|           timestamp|    state|star|count_reopen|       last_activity|
+-----------+-------------------+--------------------+--------------------+--------------------+--------------------+------------------+--------------------+-----------+--------+-------------+--------------------+---------+----+------------+--------------------+
|2021-FYJTFP|        {ความสะอาด}|          เขตบางซื่อ|             ขยะเยอะ|https://storage.g...|                NULL|100.53084,13.81865|12/14 ถนน กรุงเทพ...|       NULL|    NULL|กรุงเทพมหานคร|2021-09-03 19:51:..

---

## Applying Cleansing Pipeline

In [5]:
from src.pipelines_spark import CleansingPipelineSpark

cleansing_pipeline = CleansingPipelineSpark(spark, sedona)
df_cleansed = cleansing_pipeline.transform(df)

df_cleansed.show(10)

+-----------+-------------------+--------------------+--------------------+--------------------+-----------+--------+-------------+--------------+---------------+--------------+------------------+-------------------+------------------+---------------+---------+--------+------+
|  ticket_id|               type|        organization|             comment|             address|subdistrict|district|     province|timestamp_date|timestamp_month|timestamp_year|last_activity_date|last_activity_month|last_activity_year|resolution_time|longitude|latitude|status|
+-----------+-------------------+--------------------+--------------------+--------------------+-----------+--------+-------------+--------------+---------------+--------------+------------------+-------------------+------------------+---------------+---------+--------+------+
|2021-CGPMUN|{น้ำท่วม,ร้องเรียน}|เขตประเวศ,ฝ่ายโยธ...|น้ำท่วมเวลาฝนตกแล...|189 เฉลิมพระเกียร...|    หนองบอน|  ประเวศ|กรุงเทพมหานคร|            19|              9|    

In [6]:
df_cleansed.printSchema()

root
 |-- ticket_id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- organization: string (nullable = true)
 |-- comment: string (nullable = true)
 |-- address: string (nullable = true)
 |-- subdistrict: string (nullable = true)
 |-- district: string (nullable = true)
 |-- province: string (nullable = true)
 |-- timestamp_date: integer (nullable = true)
 |-- timestamp_month: integer (nullable = true)
 |-- timestamp_year: integer (nullable = true)
 |-- last_activity_date: integer (nullable = true)
 |-- last_activity_month: integer (nullable = true)
 |-- last_activity_year: integer (nullable = true)
 |-- resolution_time: integer (nullable = true)
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- status: string (nullable = true)



---

## Applying Model Preparation Pipeline

In [7]:
from src.pipelines_spark import ModelPrepPipelineSpark

preparing_pipeline = ModelPrepPipelineSpark()
df_prepared = preparing_pipeline.transform(df_cleansed)

df_prepared.show(10)

+---------------+--------------+---------------+--------------------+--------------------+--------------------+--------------------+
|timestamp_month|timestamp_year|resolution_time|     address_encoded|     latlong_encoded|organization_encoded|        type_encoded|
+---------------+--------------+---------------+--------------------+--------------------+--------------------+--------------------+
|              9|          2021|            275|(2048,[834,1804],...|[13.67891,100.66709]|(1786,[10,53],[1....|(25,[7,8],[1.0,1.0])|
|              9|          2021|            253|(2048,[348,426],[...| [13.7206,100.52649]|   (1786,[49],[1.0])|     (25,[14],[1.0])|
|             12|          2021|            246|(2048,[802,1656],...| [13.8228,100.59165]|(1786,[31,108],[1...|(25,[0,8],[1.0,1.0])|
|             12|          2021|            456|(2048,[802,1656],...| [13.8091,100.59131]|(1786,[31,172],[1...|      (25,[1],[1.0])|
|             12|          2021|            516|(2048,[1025,1114]...|

In [8]:
df_prepared.printSchema()

root
 |-- timestamp_month: integer (nullable = true)
 |-- timestamp_year: integer (nullable = true)
 |-- resolution_time: integer (nullable = true)
 |-- address_encoded: vector (nullable = true)
 |-- latlong_encoded: vector (nullable = true)
 |-- organization_encoded: vector (nullable = true)
 |-- type_encoded: vector (nullable = true)



---

## Model Training

In [9]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import GBTRegressor, RandomForestRegressor
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import RegressionEvaluator

# -----------------------------
# 1. Vector Assembler
# -----------------------------
assembler = VectorAssembler(
    inputCols=[
        "timestamp_month",
        "timestamp_year",
        "address_encoded",
        "latlong_encoded",
        "organization_encoded",
        "type_encoded",
    ],
    outputCol="features",
)

# -----------------------------
# 2. Regression Models
# -----------------------------
gbt = GBTRegressor(
    labelCol="resolution_time",
    featuresCol="features",
)

rf = RandomForestRegressor(
    labelCol="resolution_time",
    featuresCol="features",
)

# -----------------------------
# 3. Evaluators (Multi-Metric)
# -----------------------------
evaluators = {
    "rmse": RegressionEvaluator(
        labelCol="resolution_time", predictionCol="prediction", metricName="rmse"
    ),
    "mae": RegressionEvaluator(
        labelCol="resolution_time", predictionCol="prediction", metricName="mae"
    ),
    "r2": RegressionEvaluator(
        labelCol="resolution_time", predictionCol="prediction", metricName="r2"
    ),
}

# -----------------------------
# 4. Hyperparameter grids
# -----------------------------
gbt_paramGrid = (
    ParamGridBuilder()
    .addGrid(gbt.maxDepth, [3, 5])
    .addGrid(gbt.maxIter, [50, 100])
    .addGrid(gbt.stepSize, [0.05])
    .build()
)

rf_paramGrid = (
    ParamGridBuilder().addGrid(rf.numTrees, [100]).addGrid(rf.maxDepth, [8, 12]).build()
)

# -----------------------------
# 5. Pipelines
# -----------------------------
gbt_pipeline = Pipeline(stages=[assembler, gbt])
rf_pipeline = Pipeline(stages=[assembler, rf])

# -----------------------------
# 6. CrossValidators
# Use RMSE as main metric for tuning
# -----------------------------
cv_gbt = CrossValidator(
    estimator=gbt_pipeline,
    estimatorParamMaps=gbt_paramGrid,
    evaluator=evaluators["rmse"],
    numFolds=3,
    parallelism=1,
)

cv_rf = CrossValidator(
    estimator=rf_pipeline,
    estimatorParamMaps=rf_paramGrid,
    evaluator=evaluators["rmse"],
    numFolds=3,
    parallelism=1,
)

# -----------------------------
# 7. Train/test split
# -----------------------------
train_df, test_df = df_prepared.randomSplit([0.8, 0.2], seed=42)

In [ ]:
# -----------------------------
# 8. Train both CV models
# -----------------------------
gbt_cv_model = cv_gbt.fit(train_df)

In [ ]:
# -----------------------------
# 8. Train both CV models
# -----------------------------
rf_cv_model = cv_rf.fit(train_df)

In [ ]:
# -----------------------------
# 9. Evaluate both models on all metrics
# -----------------------------
def evaluate_model(name, model):
    print(f"\n===== {name} Results =====")
    preds = model.transform(test_df)
    for metric, evaluator in evaluators.items():
        score = evaluator.evaluate(preds)
        print(f"{metric.upper()}: {score}")
    print("==========================")


evaluate_model("GBTRegressor", gbt_cv_model)
evaluate_model("RandomForestRegressor", rf_cv_model)

In [ ]:
from pyspark.ml.tuning import CrossValidatorModel
from src.utils import get_data_dir


def save_model(model: CrossValidatorModel, model_name: str):
    save_name = f"{model_name}_best_model"
    save_path = get_data_dir() / "models" / save_name
    model.bestModel.write().overwrite().save(str(save_path))


# Overwrite best model if already existed
save_model(gbt_cv_model, "gbt")
save_model(rf_cv_model, "rf")

In [ ]:
# spark.stop()

---